In [0]:
df = spark.read.table("workspace.joint_statements_silver.joint_statements")
display(df)

In [0]:
%sql
WITH monthly_expenses AS (
SELECT 
    YEAR(date) AS `year`,
    MONTH(date) AS `month`,
    category, 
    SUM(debit) AS `outgoings`,
    SUM(credit) AS `incomings`
FROM workspace.joint_statements_silver.joint_statements
GROUP BY YEAR(date), MONTH(date), category
ORDER BY `year`, `month`, category
)
SELECT 
    `year`, 
    `month`,
    category,
    `outgoings`,
    LAG(`outgoings`, 1) OVER (
        PARTITION BY category ORDER BY `year`,`month`
        ) AS `previous_month_outgoings`,
    `outgoings` - LAG(`outgoings`, 1) OVER (
        PARTITION BY category ORDER BY `year`,`month`
        ) AS `change`
FROM monthly_expenses
ORDER BY `year`, `month`, category;


In [0]:
from pyspark.sql.functions import year, month, sum, col, lag, when
from pyspark.sql.window import Window

df = spark.read.table("workspace.joint_statements_silver.joint_statements")

# Base aggregation: Calculate monthly expenses
monthly_expenses_df = df.groupBy(
        year("date").alias("year"),
        month("date").alias("month"),
        "category"
    ).agg(
        sum("debit").alias("outgoings"),
        sum("credit").alias("incomings"),
    )     

# window specification for lag function
category_window = Window.partitionBy("category").orderBy("year", "month")

# calculate lag and change from previous
df = monthly_expenses_df.withColumn(
        "previous_outgoings",
        lag("outgoings", 1).over(category_window),
    ).withColumn(
        "change in outgoings",
        when(
            monthly_expenses_df["previous_outgoings"].isNull(), 0
        ).otherwise(
            col("outgoings") - col("previous_outgoings")
        )
    ).withColumn(
        "previous_incomings",
        lag("incomings").over(category_window),
    ).withColumn(
        "change in incomings",
        when(
            monthly_expenses_df["previous_incomings"].isNull(), 0
        ).otherwise(
            col("incomings") - col("previous_incomings")
        )
    )

display(df)



In [0]:
%sql
SELECT *
FROM workspace.joint_statements_gold.joint_statements;